### Order Book Requirements
- Maintain Bid side & Ask side (price, size)
- Tick size, Trade history
- Expected Invariants:
  - Best bid < Best Ask
  - No negative sizes
  - Prices aligned to tick

### Event processing
- Limit Order with
  * input: side=buy/sell, price, size
  * logic: adds liquidity, also if limit order crosses spread-> convert to market order
- Market Order:
  * input: side, size
  * logic: consumes relevant side liquidity, may sweep multiple price levels, moves price if best level depleted
  * Market order size > total opppositve liquidity. Todo: think!
- Cancellation
  * inpute: side, price, size
  * logic: reduces liquidity, if level zero in size-> remove price level

### Trade history recording 
Important for lambda, Trade imbalance, etc
input: side, price, size


In [6]:
"""
Assumption: no queueing right now for any price level!
Given the event processing inputs, we can take streams of order 
tuples (order_type(limit/cancel/market), side(buy/sell), price, size)

L: Limit Order
M: Market Order
X: Cancel Order

B: Buy
S: Sell
"""

events = [
    ('L', 'B', 100.05, 40),
    ('L', 'S', 100.08, 100),
    ('M', 'B', None, 50),
    ('X', 'B', 100.05, 10)
]

In [40]:
"""
raw events is not good to be stored like that. challenges: not efficient to query best bid/ask, 
size of a price also not efficient to query for adding orders/cancelling it.

Best bid: query should be fast, raw input events may take O(n) to find! We can sort it to get O(1).
Modifying limit order size: for this sorting alone don't help, because cancel order may be for not best price. So, for storing
size we can use Dict.

For adding new price level: O(N), we can limit N=20 price levels
"""
# Current state
bid_levels = {
    100.05: 40,
    100.03: 20,
    100.02: 1000
}

ask_levels = {
    100.06: 60,
    100.07: 5600,
    100.08: 100
}

bids = list(bid_levels.keys())
asks = list(ask_levels.keys())

bids.sort()
asks.sort()
print(bids,'\n', asks)

[100.02, 100.03, 100.05] 
 [100.06, 100.07, 100.08]


In [36]:
# process_events(): take events and add to data structure for bids and asks
def process_event(event):
    # event tuple (Order Type, Buy/Sell, Price, Size)
    # if type is Market order expects price = None
    order_type, side, price, size = event
    if order_type=='L' and ((side=='B' and price>=asks[0]) or (side=='S' and price<=bids[-1])):
        if side=='B':
            while size and asks:
                best_ask=asks[0]
                if best_ask>price:
                    break
                size_available = ask_levels[best_ask]
                if size>=size_available:
                    ask_levels.pop(best_ask)
                    asks.remove(best_ask)
                    size-=size_available
                else:
                    ask_levels[best_ask]-=size
                    break
        else:
            while size and bids:
                best_bid=bids[-1]
                if best_bid<price:
                    break
                size_available = bid_levels[best_bid]
                if size>=size_available:
                    bid_levels.pop(best_bid)
                    bids.remove(best_bid)
                    size-=size_available
                else:
                    bid_levels[best_bid]-=size
                    break
    elif order_type == 'L':
        if side == 'B':
            # Limit buy order
            if price in bid_levels:
                bid_levels[price]+=size
            else:
                bids.append(price)
                bids.sort()
                bid_levels[price]=size
        else: # side == 'S'
            # limit sell order
            if price in ask_levels:
                ask_levels[price]+=size
            else:
                asks.append(price)
                asks.sort()
                ask_levels[price]=size
    elif order_type == 'X':
        if side == 'B':
            # Cancel buy order
            if price in bid_levels:
                bid_levels[price]=max(bid_levels[price]-size,0) #size>=0 constraint in each level
                if bid_levels[price] == 0:
                    bid_levels.pop(price)
                    bids.remove(price)
        else: # side == 'S'
            # Cancel sell order
            if price in ask_levels:
                ask_levels[price]=max(ask_levels[price]-size,0)
                if ask_levels[price] == 0:
                    ask_levels.pop(price)
                    asks.remove(price)
    else: # order type 'M' Market Order
        if side=='B':
            while size and asks:
                best_ask=asks[0]
                size_available = ask_levels[best_ask]
                if size>=size_available:
                    ask_levels.pop(best_ask)
                    asks.remove(best_ask)
                    size-=size_available
                else:
                    ask_levels[best_ask]-=size
                    break
        else:
            while size and bids:
                best_bid=bids[-1]
                size_available = bid_levels[best_bid]
                if size>=size_available:
                    bid_levels.pop(best_bid)
                    bids.remove(best_bid)
                    size-=size_available
                else:
                    bid_levels[best_bid]-=size
                    break

In [41]:
def print_state():
    print("Bids", bids, bid_levels)
    print("Asks", asks, ask_levels)
    print("-----")
print_state()

Bids [100.02, 100.03, 100.05] {100.05: 40, 100.03: 20, 100.02: 1000}
Asks [100.06, 100.07, 100.08] {100.06: 60, 100.07: 5600, 100.08: 100}
-----


In [42]:
for event in events:
    print(event)
    process_event(event)
    print_state()

('L', 'B', 100.05, 40)
Bids [100.02, 100.03, 100.05] {100.05: 80, 100.03: 20, 100.02: 1000}
Asks [100.06, 100.07, 100.08] {100.06: 60, 100.07: 5600, 100.08: 100}
-----
('L', 'S', 100.08, 100)
Bids [100.02, 100.03, 100.05] {100.05: 80, 100.03: 20, 100.02: 1000}
Asks [100.06, 100.07, 100.08] {100.06: 60, 100.07: 5600, 100.08: 200}
-----
('M', 'B', None, 50)
Bids [100.02, 100.03, 100.05] {100.05: 80, 100.03: 20, 100.02: 1000}
Asks [100.06, 100.07, 100.08] {100.06: 10, 100.07: 5600, 100.08: 200}
-----
('X', 'B', 100.05, 10)
Bids [100.02, 100.03, 100.05] {100.05: 70, 100.03: 20, 100.02: 1000}
Asks [100.06, 100.07, 100.08] {100.06: 10, 100.07: 5600, 100.08: 200}
-----
